# Understand RAG Fundamentals

In [ ]:
# load llm 
# Create environment
import os 
from dotenv import load_dotenv 
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


# Get LLM model 
from langchain.chat_models import init_chat_model 

primary_llm = init_chat_model(model="llama-3.3-70b-versatile", model_provider="Groq")

fallback_llm_1 = init_chat_model(model="gpt-5.4-nano", model_provider="openai",
                 model_kwargs={"temperature": 0.5, "max_tokens": 1000})
fallback_llm_2 = init_chat_model(model="gpt-5.4-mini", model_provider="openai",
                 model_kwargs={"temperature": 0.5, "max_tokens": 1000})

/Users/nali/Documents/YTLLMs/.venv/lib/python3.13/site-packages/langchain/chat_models/base.py:496: UserWarning: Parameters {'max_tokens', 'temperature'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return _init_chat_model_helper(


## Select documents for RAG application

In [1]:
import requests
from langchain_core.documents import Document

from langchain_core.vectorstores import InMemoryVectorStore

from langchain_openai import OpenAIEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

DOCS_BASE = "https://docs.langchain.com"

# Curated LangChain OSS pages for this tutorial. Expand this list or parse
# URLs from https://docs.langchain.com/llms.txt to index more of the site.
DOC_PATHS = [
    "oss/python/langchain/agents",
    "oss/python/deepagents/rag",
    "oss/python/langchain/tools",
    "oss/python/langchain/models",
    "oss/python/deepagents/retrieval",
    "oss/python/langchain/knowledge-base",
    "oss/python/langchain/middleware",
    "oss/python/deepagents/overview",
    "oss/python/deepagents/subagents",
    "oss/python/deepagents/streaming",
    "oss/python/deepagents/frontend/subagent-streaming",
    "oss/python/deepagents/backends",
    "oss/python/langgraph/overview",
    "oss/python/langgraph/quickstart",
]

/Users/nali/Documents/YTLLMs/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load documents for RAG application

In [3]:
def load_langchain_docs(doc_paths: list[str] | None = None) -> list[Document]:
    """Fetch LangChain documentation pages as Documents."""
    paths = doc_paths or DOC_PATHS
    docs: list[Document] = []
    for path in paths:
        url = f"{DOCS_BASE}/{path}.md"
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
        except requests.RequestException:
            continue
        source = f"{DOCS_BASE}/{path}"
        docs.append(
            Document(page_content=response.text, metadata={"source": source})
        )
    return docs


docs = load_langchain_docs()
print(f"Loaded {len(docs)} documentation pages.")

Loaded 14 documentation pages.


## Split the documents

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
print(f"No. of splits: {len(all_splits)}")

No. of splits: 939


## Add your embedding model 

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    encode_kwargs={"normalize_embeddings": True,
            'batch_size':32
                   },
    
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19703.19it/s]


In [16]:
vector = embeddings.embed_query("Hi")
print(len(vector))
#print(vector)

768


## Create a vector store 

In [17]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

In [77]:
vector_store.add_documents(documents=all_splits)
print(f"Indexed {len(all_splits)} chunks.")

Indexed 939 chunks.


### Do some Semantic Search testing 

In [19]:
vector_store.similarity_search(query="What is Langchain agent?",k=1)

[Document(id='0de987b9-42a0-4d67-b6f1-87c548abce3d', metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/overview'}, page_content='<Expandable title="how LangChain products fit together" defaultOpen={false}>\n  * [Deep Agents](/oss/python/deepagents/overview) is an [agent harness](/oss/python/concepts/products#agent-harnesses-like-the-deep-agents-sdk): planning, subagents, filesystem tools, and context management on top of LangGraph.\n  * [LangChain](/oss/python/langchain/overview) is the agent framework: abstractions and integrations for models, tools, and agent loops.\n  * [LangGraph](/oss/python/langgraph/overview) is the orchestration runtime: durable execution, streaming, human-in-the-loop, and persistence.\n  * [LangSmith](/langsmith/observability) is the platform for tracing, evaluation, prompts, and deployment across frameworks.\n  * [LangSmith Engine](/langsmith/engine) detects issues in your LangGraph agent traces and proposes fixes. You can open a pull reque

In [29]:
results = vector_store.similarity_search_with_score(query="What is Langchain agent?",k=5)

for result in results: 
    doc, score = result
    print(score)
    print(doc.page_content[:100])


0.5610599517822266
<Expandable title="how LangChain products fit together" defaultOpen={false}>
  * [Deep Agents](/oss/
0.6833313703536987
[LangChain](/oss/python/langchain/) is the framework that provides the core building blocks for your
0.6939014196395874
<Card title="LangChain" icon="https://mintcdn.com/langchain-5e9cc07a/nQm-sjd_MByLhgeW/images/brand/l
0.7251949310302734
* [LangSmith Engine](/langsmith/engine) detects issues in your LangGraph agent traces and proposes f
0.7416951656341553
> ## Documentation Index
> Fetch the complete documentation index at: https://docs.langchain.com/llm


## Retriever interface 

In [42]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)

In [40]:
retriever.invoke(input="How to create Langgrapgh agents?")

[Document(id='0866a356-ef7f-4a8c-9ac1-631c6cf73309', metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/quickstart'}, page_content='<Tip>\n      Trace and debug your agent with [LangSmith](https://smith.langchain.com?utm_source=docs\\&utm_medium=cta\\&utm_campaign=langsmith-signup\\&utm_content=oss-langgraph-quickstart). Follow the [tracing quickstart](/langsmith/trace-with-langgraph) to get set up. When ready for production, see [Deploy](/langsmith/deployment) for hosting options.\n\n      We recommend you also set up [LangSmith Engine](/langsmith/engine) which monitors your traces, detects issues, and proposes fixes.\n    </Tip>\n\n    Congratulations! You\'ve built your first agent using the LangGraph Functional API.\n\n    <Accordion title="Full code example" icon="code">\n      ```python theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}\n      # Step 1: Define tools and model\n\n      from langchain.tools import tool\n      from langchain.cha

In [43]:
retriever.batch([
    "What is an agent", "What is a deep agent"
])

[[Document(id='dd9babb0-5e45-4b3d-bc01-957cd882b758', metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/overview'}, page_content="[LangChain](/oss/python/langchain/) is the framework that provides the core building blocks for your agents.\nTo learn more about the differences between LangChain, LangGraph, and Deep Agents, see [Frameworks, runtimes, and harnesses](/oss/python/concepts/products). For a side-by-side comparison with Anthropic's harness, see [Deep Agents vs. Claude Agent SDK](/oss/python/deepagents/comparison).\n\nFor building custom agents without these built-in capabilities, consider using LangChain's [`create_agent`](/oss/python/langchain/agents) or building a custom [LangGraph](/oss/python/langgraph/overview) workflow.\n\n## Execution environment\n\nThe execution environment is where an agent acts. It has four layers:")],
 [Document(id='4d9fa4d2-0e22-4b4b-8d04-e3355ac10035', metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/streami

# Retriver as a tool 

In [55]:
from langchain_core.tools import tool 

@tool 
def get_documents(query:str)->str: 
    """This tool will return the retrieved information"""
    docs = retriever.invoke(query)

    return "\n\n".join(doc.page_content for doc in docs)
    

## Add tool with standalone LLM

In [56]:
llm_with_rag_tool = primary_llm.bind_tools([get_documents])

In [57]:
llm_with_rag_tool

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x12482d2b0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12482dfd0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_documents', 'description': 'This tool will return t

In [64]:
from langchain.messages import HumanMessage, SystemMessage
result = llm_with_rag_tool.invoke(
    [
        SystemMessage(content="You are an helpful assistant"),
        HumanMessage(content="How to cretae a langchain agent assistant?")
    ]
)

tool = result.tool_calls[0]
tool

{'name': 'get_documents',
 'args': {'query': 'create LangChain agent assistant'},
 'id': 'b0fktq3zj',
 'type': 'tool_call'}

In [65]:
get_documents.invoke(tool)

ToolMessage(content="> ## Documentation Index\n> Fetch the complete documentation index at: https://docs.langchain.com/llms.txt\n> Use this file to discover all available pages before exploring further.\n\n# Quickstart\n\nThis quickstart demonstrates how to build a calculator agent using the LangGraph Graph API or the Functional API.\n\n<Tip>\n  **Using an AI coding assistant?**\n\n  * Install the [LangChain Docs MCP server](/use-these-docs) to give your agent access to up-to-date LangChain documentation and examples.\n  * Install [LangChain Skills](https://github.com/langchain-ai/langchain-skills) to improve your agent's performance on LangChain ecosystem tasks.\n</Tip>\n\n* [Use the Graph API](#use-the-graph-api) if you prefer to define your agent as a graph of nodes and edges.\n* [Use the Functional API](#use-the-functional-api) if you prefer to define your agent as a single function.", name='get_documents', tool_call_id='b0fktq3zj')

## Add tool with Langchain Agent

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=primary_llm, 
    tools=[get_documents],
    system_prompt="""Use get documents tool for answering agent, deep agent, langchain and langgraph related queries."""
)

In [71]:
results = agent.invoke({
    "messages":[
        SystemMessage(content="You are an helpful assistant"),
                HumanMessage(content="How to cretae a langchain agent assistant?")
    ]
})

In [76]:
print(results["messages"][-1].content)

To create a LangChain agent assistant, you can follow these steps:

1. **Define the agent's goals and objectives**: Determine what tasks the agent should perform and what kind of assistance it should provide.
2. **Choose a programming language**: LangChain supports multiple programming languages, including Python, JavaScript, and TypeScript. Choose the language that best fits your needs.
3. **Set up the LangChain framework**: Install the LangChain library and set up the framework according to the documentation.
4. **Define the agent's architecture**: Design the agent's architecture, including the components, modules, and APIs that will be used.
5. **Implement the agent's logic**: Write the code that defines the agent's behavior, including the decision-making processes, actions, and interactions with the environment.
6. **Integrate with models and tools**: Integrate the agent with relevant models, such as language models, and tools, such as databases or APIs.
7. **Test and refine the ag